# Feature Engineering and Modeling

---

### Goal
Build customer-level features using a temporal split approach,
train three models, and compare their performance honestly.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from xgboost import XGBClassifier
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

print("Libraries ready.")

Libraries ready.


In [2]:
orders_enriched = pd.read_csv('../data/processed/orders_enriched.csv',
                               parse_dates=['order_purchase_timestamp'])
products        = pd.read_csv('../data/raw/olist_products_dataset.csv')
items           = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
reviews         = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
payments        = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')

print(f"orders_enriched: {orders_enriched.shape}")
print(f"\nDate range in dataset:")
print(f"  Earliest: {orders_enriched['order_purchase_timestamp'].min().date()}")
print(f"  Latest:   {orders_enriched['order_purchase_timestamp'].max().date()}")

orders_enriched: (96478, 11)

Date range in dataset:
  Earliest: 2016-09-15
  Latest:   2018-08-29


---
## Sprint 5 — Feature Engineering

### The Temporal Split Approach

We use two separate time windows to prevent data leakage:

FEATURE WINDOW → all purchases before March 2018
  Used to build: recency, frequency, monetary, behavioral features

CHURN WINDOW → March 2018 to June 2018 (90 days)
  Used to define: did this customer buy again or not?

Because features and labels come from different time periods,
recency is now a safe feature — it measures past behavior,
not the future behavior that defines the label.

In [3]:
OBS_DATE        = pd.Timestamp('2018-03-01')
CHURN_WINDOW_END = OBS_DATE + pd.Timedelta(days=90)

orders_before = orders_enriched[
    orders_enriched['order_purchase_timestamp'] < OBS_DATE
].copy()

orders_during = orders_enriched[
    (orders_enriched['order_purchase_timestamp'] >= OBS_DATE) &
    (orders_enriched['order_purchase_timestamp'] <  CHURN_WINDOW_END)
].copy()

print(f"Observation date:       {OBS_DATE.date()}")
print(f"Churn window end:       {CHURN_WINDOW_END.date()}")
print(f"\nOrders in feature window:  {len(orders_before):,}")
print(f"Orders in churn window:    {len(orders_during):,}")
print(f"\nCustomers with history:    {orders_before['customer_unique_id'].nunique():,}")
print(f"Customers who returned:    {orders_during['customer_unique_id'].nunique():,}")

Observation date:       2018-03-01
Churn window end:       2018-05-30

Orders in feature window:  57,319
Orders in churn window:    20,280

Customers with history:    55,525
Customers who returned:    19,988


In [4]:
rfm = orders_before.groupby('customer_unique_id').agg(
    recency   = ('order_purchase_timestamp', lambda x: (OBS_DATE - x.max()).days),
    frequency = ('order_id', 'count'),
    monetary  = ('order_value', 'sum')
).reset_index()

rfm['avg_order_value'] = rfm['monetary'] / rfm['frequency']

print(f"RFM table shape: {rfm.shape}")
print(rfm[['recency', 'frequency', 'monetary', 'avg_order_value']].describe().round(2))

RFM table shape: (55525, 5)
        recency  frequency  monetary  avg_order_value
count  55525.00   55525.00  55525.00         55525.00
mean     156.83       1.03    162.35           157.73
std      112.25       0.20    222.46           215.75
min        0.00       1.00      0.00             0.00
25%       63.00       1.00     62.53            61.78
50%      133.00       1.00    106.19           104.21
75%      239.00       1.00    179.70           174.43
max      531.00       9.00  13664.08         13664.08


In [5]:
lifespan = orders_before.groupby('customer_unique_id').agg(
    first_purchase = ('order_purchase_timestamp', 'min'),
    last_purchase  = ('order_purchase_timestamp', 'max')
).reset_index()

lifespan['customer_lifespan_days'] = (
    lifespan['last_purchase'] - lifespan['first_purchase']
).dt.days

lifespan = lifespan[['customer_unique_id', 'customer_lifespan_days']]
rfm = rfm.merge(lifespan, on='customer_unique_id', how='left')

print(f"Shape after lifespan: {rfm.shape}")
print(f"customer_lifespan_days sample:")
print(rfm['customer_lifespan_days'].describe().round(2))

Shape after lifespan: (55525, 6)
customer_lifespan_days sample:
count    55525.00
mean         1.60
std         16.71
min          0.00
25%          0.00
50%          0.00
75%          0.00
max        454.00
Name: customer_lifespan_days, dtype: float64


In [6]:
items_with_cat = items.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

items_with_customer = items_with_cat.merge(
    orders_before[['order_id', 'customer_unique_id']],
    on='order_id',
    how='inner'
)

category_features = (
    items_with_customer
    .groupby('customer_unique_id')
    .agg(
        unique_categories = ('product_category_name', 'nunique'),
        total_items       = ('order_id', 'count')
    )
    .reset_index()
)

rfm = rfm.merge(category_features, on='customer_unique_id', how='left')

print(f"Shape after category features: {rfm.shape}")
print(rfm[['unique_categories', 'total_items']].describe().round(2))

Shape after category features: (55525, 8)
       unique_categories  total_items
count            55525.0     55525.00
mean                 1.0         1.18
std                  0.2         0.61
min                  0.0         1.00
25%                  1.0         1.00
50%                  1.0         1.00
75%                  1.0         1.00
max                  5.0        21.00


In [7]:
reviews_clean = (
    reviews[['order_id', 'review_score']]
    .drop_duplicates('order_id')
)

reviews_with_customer = (
    orders_before[['order_id', 'customer_unique_id']]
    .merge(reviews_clean, on='order_id', how='left')
)

review_features = (
    reviews_with_customer
    .groupby('customer_unique_id')
    .agg(
        avg_review_score = ('review_score', 'mean'),
        review_count     = ('review_score', 'count')
    )
    .reset_index()
)

rfm = rfm.merge(review_features, on='customer_unique_id', how='left')

print(f"Shape after review features: {rfm.shape}")
print(rfm[['avg_review_score', 'review_count']].describe().round(2))

Shape after review features: (55525, 10)
       avg_review_score  review_count
count          55118.00      55525.00
mean               4.13          1.02
std                1.29          0.22
min                1.00          0.00
25%                4.00          1.00
50%                5.00          1.00
75%                5.00          1.00
max                5.00          9.00


In [8]:
payments_with_customer = (
    orders_before[['order_id', 'customer_unique_id']]
    .merge(
        payments[['order_id', 'payment_type', 'payment_installments']],
        on='order_id',
        how='left'
    )
)

payment_features = (
    payments_with_customer
    .groupby('customer_unique_id')
    .agg(
        avg_installments = ('payment_installments', 'mean'),
        used_credit_card = ('payment_type', lambda x: int((x == 'credit_card').any()))
    )
    .reset_index()
)

rfm = rfm.merge(payment_features, on='customer_unique_id', how='left')

print(f"Shape after payment features: {rfm.shape}")
print(rfm[['avg_installments', 'used_credit_card']].describe().round(2))

Shape after payment features: (55525, 12)
       avg_installments  used_credit_card
count          55524.00          55525.00
mean               2.97              0.77
std                2.73              0.42
min                1.00              0.00
25%                1.00              1.00
50%                2.00              1.00
75%                4.00              1.00
max               24.00              1.00


In [9]:
customers_with_history = orders_before['customer_unique_id'].unique()
customers_who_returned = set(orders_during['customer_unique_id'].unique())

churn_labels = pd.DataFrame({
    'customer_unique_id': customers_with_history,
    'churned': [
        0 if c in customers_who_returned else 1
        for c in customers_with_history
    ]
})

total   = len(churn_labels)
churned = churn_labels['churned'].sum()
active  = total - churned

print(f"Total customers with purchase history: {total:,}")
print(f"Churned — did not return:  {churned:,}  ({churned/total:.1%})")
print(f"Active  — did return:      {active:,}   ({active/total:.1%})")

Total customers with purchase history: 55,525
Churned — did not return:  55,139  (99.3%)
Active  — did return:      386   (0.7%)


In [10]:
rfm = rfm.merge(churn_labels, on='customer_unique_id', how='inner')

rfm['avg_review_score']       = rfm['avg_review_score'].fillna(rfm['avg_review_score'].median())
rfm['review_count']           = rfm['review_count'].fillna(0)
rfm['avg_installments']       = rfm['avg_installments'].fillna(1)
rfm['unique_categories']      = rfm['unique_categories'].fillna(1)
rfm['total_items']            = rfm['total_items'].fillna(1)
rfm['used_credit_card']       = rfm['used_credit_card'].fillna(0)
rfm['customer_lifespan_days'] = rfm['customer_lifespan_days'].fillna(0)

print(f"Final feature table shape: {rfm.shape}")
print(f"Missing values remaining:  {rfm.isnull().sum().sum()}")
print(f"Churn rate:                {rfm['churned'].mean():.1%}")
print(f"\nAll columns:")
for col in rfm.columns:
    print(f"  {col}")

rfm.to_csv('../data/processed/features_and_labels.csv', index=False)
print("\nSaved to data/processed/features_and_labels.csv")

Final feature table shape: (55525, 13)
Missing values remaining:  0
Churn rate:                99.3%

All columns:
  customer_unique_id
  recency
  frequency
  monetary
  avg_order_value
  customer_lifespan_days
  unique_categories
  total_items
  avg_review_score
  review_count
  avg_installments
  used_credit_card
  churned

Saved to data/processed/features_and_labels.csv
